**Numeriser le df avant de passer dans le pipeline de nettoyage pour que l'etape des correlations prenne en compte les variables qualitatives**


In [ ]:
import sys
sys.path.append('..')

import pandas as pd
from utils.fonctions import load_parquet_data, load_pickle
from src.artifacts.data_processing import Nettoyage

def numerize(data, encoding_pipeline, downcast_num_cols=True):
    """
    data : pd.DataFrame, dataframe a numeriser
    encoding_pipeline : pipeline de numerisation des variables categorielles
    downcast_num_cols : bool, si True, downcast les colonnes numeriques
    """
    def downcast_numeric_cols():
        # int32 range = 2**32/2 à gauche et à droite de 0
        for c in data.columns:
            if data[c].dtype == 'int64':
                data[c] = data[c].astype('int32')
            elif data[c].dtype == 'float64':
                data[c] = data[c].astype('float32')
        return data
    
    encoding_pipeline.set_output(transform='pandas')
    data = encoding_pipeline.fit_transform(data)

    if downcast_num_cols:
        return downcast_numeric_cols()
    return data

In [2]:
categ_encoders = load_pickle("../ressources/encoders_fitted/categorical_encoders.pkl")
df = load_parquet_data('../ressources/data/2_intermediary/enedis_ban_ademe_extract_PARIS_2022.parquet')

Loading parquet data from : ../ressources/data/2_intermediary/enedis_ban_ademe_extract_PARIS_2022.parquet..


In [3]:
df['arrondissement'] = 1
numerized_df = numerize(df, categ_encoders)

In [4]:
pipeline = Nettoyage(numerized_df)

pipeline.run(use_entropy_selection = True)
df_clean_v1 = pipeline.df

print(f'il reste {len(df_clean_v1.columns)} colonnes et {len(df_clean_v1)} lignes.')
df_clean_v1.info()

-> Delete cols mano..
-> Delete NaN cols..
Le dataframe contient initialement 251 colonnes et 464576 lignes.
Start processing : columns identification..
Il y a 76 colonnes vides à au moins, 90.0%, 63 colonnes avec une entropie de 0 ou 1 et 13 colonnes inutiles
Start processing : columns deleting ...
Il reste 138 colonnes et 370056 lignes.
-> Start object columns auto casting..
End casting
-> Fill pd.NA for float dtypes..
Done fillnan !
Les colonnes suivantes ont une corrélation supérieure à 0.9 : ['deperditions_enveloppe_ademe', 'cout_chauffage_energie_ndeg1_ademe', 'cout_chauffage_ademe', 'emission_ges_chauffage_energie_ndeg1_ademe', 'emission_ges_ecs_depensier_ademe', 'apports_internes_saison_chauffe_ademe', 'conso_auxiliaires_e_finale_ademe', 'emission_ges_ecs_ademe', 'cout_ecs_energie_ndeg1_ademe', 'cout_ecs_depensier_ademe', 'conso_chauffage_installation_chauffage_ndeg1_ademe', 'emission_ges_chauffage_depensier_ademe', 'emission_ges_auxiliaires_ademe', 'deperditions_ponts_thermiqu

In [5]:
pipeline = Nettoyage(df)

pipeline.run(use_target_correlation_selection = True)
df_clean_v2 = pipeline.df

print(f'il reste {len(df_clean_v2.columns)} colonnes et {len(df_clean_v2)} lignes.')
df_clean_v1.info()

-> Delete cols mano..
-> Delete NaN cols..
Le dataframe contient initialement 251 colonnes et 464576 lignes.
Start processing : columns identification..
Il y a 76 colonnes vides à au moins, 90.0%, 56 colonnes avec une entropie de 0 ou 1 et 13 colonnes inutiles
Start processing : columns deleting ...
Il reste 145 colonnes et 375231 lignes.
-> Start object columns auto casting..
End casting
-> Fill pd.NA for float dtypes..
Done fillnan !
Les colonnes suivantes ont une corrélation supérieure à 0.9 : ['deperditions_enveloppe_ademe', 'conso_e_finale_depensier_installation_ecs_ademe', 'cout_chauffage_energie_ndeg1_ademe', 'conso_ecs_e_finale_energie_ndeg2_ademe', 'cout_chauffage_ademe', 'emission_ges_chauffage_energie_ndeg1_ademe', 'cout_refroidissement_depensier_ademe', 'emission_ges_chauffage_energie_ndeg2_ademe', 'emission_ges_ecs_depensier_ademe', 'apports_internes_saison_chauffe_ademe', 'conso_auxiliaires_e_finale_ademe', 'emission_ges_ecs_ademe', 'cout_ecs_depensier_ademe', 'conso_chau

In [6]:
set(df_clean_v1.columns) - set(df_clean_v2.columns) # cols dans v1 mais pas dans v2

{'besoin_chauffage_ademe',
 'categorie_enr_ademe_0',
 'configuration_installation_chauffage_ndeg1_ademe_0',
 'configuration_installation_ecs_ademe_0',
 'conso_chauffage_e_primaire_ademe',
 'conso_e_finale_depensier_installation_ecs_ademe',
 'conso_ecs_depensier_e_primaire_ademe',
 'conso_ecs_e_finale_energie_ndeg2_ademe',
 'conso_refroidissement_e_finale_ademe',
 'cout_refroidissement_ademe',
 'cout_refroidissement_depensier_ademe',
 'emission_ges_chauffage_energie_ndeg2_ademe',
 'methode_application_dpe_ademe_0',
 'type_batiment_ademe_0',
 'type_emetteur_installation_chauffage_ndeg1_ademe_0',
 'type_enedis_with_ban_0',
 'type_energie_generateur_ecs_ndeg1_ademe_0',
 'type_energie_generateur_ndeg1_installation_ndeg1_ademe_0',
 'type_energie_ndeg1_ademe_0',
 'type_energie_ndeg2_ademe_0',
 'type_energie_principale_chauffage_ademe_0',
 'type_energie_principale_ecs_ademe_0',
 'type_generateur_ecs_ndeg1_ademe_0',
 'type_generateur_ndeg1_installation_ndeg1_ademe_0',
 'type_installation_chauff

In [7]:
set(df_clean_v2.columns) - set(df_clean_v1.columns) # cols dans v2 mais pas dans v1

{'appartement_non_visite_0_1_ademe',
 'besoin_ecs_ademe',
 'categorie_enr_ademe',
 'configuration_installation_chauffage_ndeg1_ademe',
 'configuration_installation_ecs_ademe',
 'conso_chauffage_e_finale_energie_ndeg2_ademe',
 'conso_e_finale_installation_ecs_ademe',
 'conso_ecs_e_primaire_ademe',
 'conso_refroidissement_depensier_e_finale_ademe',
 'conso_refroidissement_depensier_e_primaire_ademe',
 'cout_ecs_energie_ndeg1_ademe',
 'cout_ecs_energie_ndeg2_ademe',
 'cout_total_5_usages_energie_ndeg1_ademe',
 'deperditions_ponts_thermiques_ademe',
 'indicateur_confort_ete_ademe',
 'inertie_lourde_0_1_ademe',
 'isolation_toiture_0_1_ademe',
 'logement_traversant_0_1_ademe',
 'methode_application_dpe_ademe',
 'presence_brasseur_air_0_1_ademe',
 'protection_solaire_exterieure_0_1_ademe',
 'type_batiment_ademe',
 'type_emetteur_installation_chauffage_ndeg1_ademe',
 'type_enedis_with_ban',
 'type_energie_generateur_ecs_ndeg1_ademe',
 'type_energie_generateur_ndeg1_installation_ndeg1_ademe',
 

**récap fonction pour executer tout ca en un seul**

In [19]:
def run_nettoyage(data, encoding_pipeline, nettoyage_pipeline: Nettoyage, methode='entropy', downcast_num_cols=True) -> Nettoyage:
    """
    data : pd.DataFrame, dataframe a numeriser
    encoding_pipeline : pipeline de numerisation des variables categorielles
    nettoyage_pipeline : pipeline de nettoyage des variables numeriques (importer depuis src.artifacts.data_processing)
    methode : Methode de nettoyage a utiliser : 'entropy' ou 'target_correlation'
    downcast_num_cols : bool, si True, downcast les colonnes numeriques
    """
    
    def downcast_numeric_cols(data):
        # int32 range = 2**32/2 à gauche et à droite de 0
        for c in data.columns:
            if data[c].dtype == 'int64':
                data[c] = data[c].astype('int32')
            elif data[c].dtype == 'float64':
                data[c] = data[c].astype('float32')
        return data
    
    def numerize(data, encoding_pipeline, downcast_num_cols):
        encoding_pipeline.set_output(transform='pandas')
        data = encoding_pipeline.fit_transform(data)
        if downcast_num_cols:
            print(f"-> downcast des colonnes numeriques (64->32)")
            data = downcast_numeric_cols(data)
        return data

    if 'arrondissement' not in data.columns: data['arrondissement'] = 1
    data = numerize(data, encoding_pipeline, downcast_num_cols)
    pipeline_ins = nettoyage_pipeline(data)
    if methode == 'entropy':
        pipeline_ins.run(use_entropy_selection=True)
    elif methode == 'target_correlation':
        pipeline_ins.run(use_target_correlation_selection=True)
    else:
        raise ValueError("La méthode doit être 'entropy' ou 'target_correlation'")
    return pipeline_ins

In [20]:
categ_encoders = load_pickle("../ressources/encoders_fitted/categorical_encoders.pkl")
df = load_parquet_data('../ressources/data/2_intermediary/enedis_ban_ademe_extract_PARIS_2022.parquet')

res = run_nettoyage(df, categ_encoders, Nettoyage, methode='entropy', downcast_num_cols=True)
res.df.head()

Loading parquet data from : ../ressources/data/2_intermediary/enedis_ban_ademe_extract_PARIS_2022.parquet..
-> downcast des colonnes numeriques (64->32)
-> Delete cols mano..
-> Delete NaN cols..
Le dataframe contient initialement 251 colonnes et 464576 lignes.
Start processing : columns identification..
Il y a 76 colonnes vides à au moins, 90.0%, 63 colonnes avec une entropie de 0 ou 1 et 13 colonnes inutiles
Start processing : columns deleting ...
Il reste 138 colonnes et 370056 lignes.
-> Start object columns auto casting..
End casting
-> Fill pd.NA for float dtypes..
Done fillnan !
Les colonnes suivantes ont une corrélation supérieure à 0.9 : ['deperditions_enveloppe_ademe', 'cout_chauffage_energie_ndeg1_ademe', 'cout_chauffage_ademe', 'emission_ges_chauffage_energie_ndeg1_ademe', 'emission_ges_ecs_depensier_ademe', 'apports_internes_saison_chauffe_ademe', 'conso_auxiliaires_e_finale_ademe', 'emission_ges_ecs_ademe', 'cout_ecs_energie_ndeg1_ademe', 'cout_ecs_depensier_ademe', 'cons

,qualite_isolation_menuiseries_ademe,qualite_isolation_murs_ademe,qualite_isolation_plancher_haut_comble_perdu_ademe,qualite_isolation_enveloppe_ademe,qualite_isolation_plancher_bas_ademe,qualite_isolation_plancher_haut_toit_terrase_ademe,etiquette_ges_ademe,etiquette_dpe_ademe,classe_inertie_batiment_ademe,periode_construction_ademe,...,cout_auxiliaires_ademe,score_ban_ademe,surface_habitable_immeuble_ademe,annee_construction_ademe,nombre_niveau_immeuble_ademe,consommation_annuelle_totale_de_l_adresse_mwh_enedis_with_ban,lon_enedis_with_ban,lat_enedis_with_ban,arrondissement,conso_kwh_m2
0,4,1.0,4.0,1,4.0,-1.0,3,3,2.0,1,...,111.900002,0.72,2554.5,1947.0,7.0,78.292,2.328325,48.84874,6,22.762951
1,1,1.0,1.0,4,4.0,-1.0,2,4,2.0,1,...,0.000000,0.72,2554.5,1947.0,7.0,78.292,2.328325,48.84874,6,212.195129
2,1,1.0,4.0,1,4.0,-1.0,4,5,2.0,1,...,138.100006,0.72,2554.5,1947.0,7.0,78.292,2.328325,48.84874,6,22.550545
3,2,1.0,1.0,3,4.0,-1.0,3,6,2.0,1,...,0.000000,0.72,2554.5,1947.0,7.0,78.292,2.328325,48.84874,6,214.285721
4,3,1.0,3.0,4,4.0,-1.0,3,6,2.0,1,...,0.000000,0.72,2554.5,1947.0,7.0,78.292,2.328325,48.84874,6,231.382996
